# Imports

In [0]:
#init

from pyspark.sql import functions as F
from pyspark.sql.functions import when, upper, lower, trim, col
from pyspark.sql.types import StringType, IntegerType

# Read data from the bronze_schema

In [0]:
df = spark.table("workspace.bronze_schema.erp_px_cat_g1v2")
df.display()

# Explore data

In [0]:
# print schema
df.printSchema

In [0]:
# check data information
df.describe().display()

"""
    All column counts are equal meaning data might be complete
    To be explored further
"""

## Check for white spaces

In [0]:
for field in df.schema:
    column_name = field.name
    column_data_type = field.dataType

    if isinstance(column_data_type, StringType):
        white_space_df = df.select("*").where(col(column_name).isNull())
        white_space_count = white_space_df.count()
        print(f"{white_space_count} white spaces found in column {column_name}")

        if white_space_count:
            
            white_space_df.display()
    

"""
    No white space found
"""

## Check for null values

In [0]:
for column in df.columns: 
    null_df = df.select("*").where(col(column).isNull())
    null_df_count = null_df.count()
    print(f"{null_df_count} null values found in Column {column}")

    if null_df_count:
        
        null_df.display()

"""
    No null values found
"""

## Check for distinct values in Columns cat, subcat, and maintanance

In [0]:
# Check distinct values 
df.select("CAT").distinct().display()  

# Check distinct values 
df.select("SUBCAT").distinct().display()  

# Check distinct values 
df.select("MAINTENANCE").distinct().display()  


"""
  All distinct values look good
"""

# Data Notes
- Column needs to be renamed
- Data is clean.. No transformation needed

# Data Transformation

## Column remapping

In [0]:
COLUMNS_RENAMED = {
    'ID': 'category_id',
    'CAT': 'category',
    'SUBCAT': 'sub_category',
    'MAINTENANCE': 'maintenance'
}

df = df.withColumnsRenamed(COLUMNS_RENAMED)

df.display()

# Write to silver_schema

In [0]:
df.write.mode('overwrite').format('delta').saveAsTable("workspace.silver_schema.erp_px_cat_g1v2")

# Read loaded data from silver_schema

In [0]:
silver_erp_px_cat_g1v2_df = spark.table("workspace.silver_schema.erp_px_cat_g1v2")
silver_erp_px_cat_g1v2_df.display()